# Titanic survival — model code

Data cleaning and feature engineering, preprocessing, model training and evaluation.

Historical feature choices were informed by full-dataset EDA. Models reuse the same evaluation subset, so the results are exploratory, with no independent final evaluation. [Project Documentation](Project%20Documentation/README.md).

**Code assistance:** The code was developed with assistance from **CODEX**.


## 1. Load


In [ ]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


df = pd.read_csv("Data/Titanic_dataset.csv")


## 2. Clean and derive features


In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")

df["Title"] = df["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

main_titles = ["Mr", "Mrs", "Miss", "Master"]

df.loc[~df["Title"].isin(main_titles), "Title"] = "Rare"

df = df.dropna(subset=["Embarked"])

df = df.drop(columns=[
    "PassengerId",
    "Name",
    "Ticket",
    "Cabin"
])


## 3. Split


In [ ]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


## 4. Preprocess


In [ ]:
numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Title"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


## 5. Train and select models


In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42
    ))
])

random_forest_model.fit(X_train, y_train)

rf_param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 5, 10],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2]
}

rf_grid_search = GridSearchCV(
    estimator=random_forest_model,
    param_grid=rf_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

rf_grid_search.fit(X_train, y_train)

print("Selected Random Forest parameters:", rf_grid_search.best_params_)
print(f"Cross-validation ROC AUC: {rf_grid_search.best_score_:.4f}")


## 6. Evaluate


In [ ]:
def evaluate_model(name, estimator):
    predictions = estimator.predict(X_test)
    probabilities = estimator.predict_proba(X_test)[:, 1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC AUC": roc_auc_score(y_test, probabilities),
    }


results = pd.DataFrame([
    evaluate_model("Logistic Regression", model),
    evaluate_model("Random Forest baseline", random_forest_model),
    evaluate_model("Random Forest selected by CV", rf_grid_search.best_estimator_),
]).set_index("Model")
results.round(4)
